In [1]:
!pip install --quiet spacy pandas
!python -m spacy download ja_core_news_sm   # 日本語モデル (~50 MB)


[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "/Users/teshima/lecture-supplement-material/Introduction-to-AI/.venv/lib/python3.11/site-packages/spacy/__init__.py", line 6, in <module>
    from .errors import setup_default_warnings
  File "/Users/teshima/lecture-supplement-materia

In [2]:
# -----------------------------------------
# 0. キャッシュ保存先の指定  ★ここだけ編集すればOK
# -----------------------------------------
import os, pathlib
CACHE_DIR = "./cache"          # ← 好きなパスに変更
pathlib.Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

os.environ["XDG_CACHE_HOME"] = CACHE_DIR
os.environ["SPACY_HOME"]      = os.path.join(CACHE_DIR, "spacy")
os.environ["HF_HOME"]         = os.path.join(CACHE_DIR, "huggingface")

# -----------------------------------------
# 1. Imports & Setup
# -----------------------------------------
import subprocess, sys, importlib
import numpy as np
import pandas as pd
import spacy
from IPython.display import display, Markdown

np.random.seed(0)   # reproducibility

# -----------------------------------------
# 2. spaCy モデルの確実なロード
# -----------------------------------------
MODEL_NAME = "ja_core_news_sm"

def ensure_spacy_model(model_name: str):
    try:
        importlib.import_module(model_name)
    except ImportError:
        print(f"[INFO] spaCy model '{model_name}' not found — downloading...")
        subprocess.run(
            [sys.executable, "-m", "spacy", "download", model_name, "--quiet"],
            check=True,
        )

ensure_spacy_model(MODEL_NAME)
nlp = spacy.load(MODEL_NAME)

# -----------------------------------------
# 3. デモ文（DATE・PERSON・GPE を含む）
# -----------------------------------------
text = "6月27日、ハルは国立科学博物館の展示を訪れた。"
doc = nlp(text)

# -----------------------------------------
# 4. 品詞タグ付け (POS tagging) ― 縦長テーブル
# -----------------------------------------
pos_df = pd.DataFrame({
    "Token": [tok.text for tok in doc],
    "Lemma": [tok.lemma_ for tok in doc],
    "POS":   [tok.pos_   for tok in doc],
    "Tag":   [tok.tag_   for tok in doc],
    "Dep":   [tok.dep_   for tok in doc],
})

display(Markdown("### 品詞タグ付け結果（縦長）"))
display(
    pos_df.style.set_properties(
        **{
            "background-color": "#E0FFFF",  # light_light_blue
            "border-color":     "#D3D3D3",  # light_gray
        }
    )
)

# -----------------------------------------
# 5. 固有表現認識 (NER) ― エンティティ単位
# -----------------------------------------
ner_df = pd.DataFrame({
    "Entity": [ent.text for ent in doc.ents],
    "Label":  [ent.label_ for ent in doc.ents],
    "Start":  [ent.start_char for ent in doc.ents],
    "End":    [ent.end_char   for ent in doc.ents],
})

display(Markdown("### 固有表現認識結果（エンティティ単位）"))
display(ner_df if not ner_df.empty else Markdown("*No named entities detected.*"))

# -----------------------------------------
# 6. 系列ラベリング中間表（Token / POS / NER の縦長）
# -----------------------------------------
seq_df = pd.DataFrame({
    "Token": [tok.text for tok in doc],
    "POS":   [tok.pos_  for tok in doc],
    "NER":   [
        f"{tok.ent_iob_}-{tok.ent_type_}" if tok.ent_type_ else "O"
        for tok in doc
    ],
    # スパンを "start-end" で表現
    "Span": [f"{tok.idx}-{tok.idx+len(tok)}" for tok in doc],
})

# -----------------------------------------
# 7. 横長テーブル（Token が列、行に POS・NER・Span）
#    * スライドに貼り付けやすい形
# -----------------------------------------
wide_df = pd.DataFrame(
    [seq_df["Token"].to_list(),
     seq_df["POS"].to_list(),
     seq_df["NER"].to_list(),
     seq_df["Span"].to_list()],
    index=["Token", "POS", "NER", "Span"]
)

wide_md = wide_df.to_markdown()

display(Markdown("### 系列ラベリング結果（横長: 列がトークン、行が属性＋スパン）"))
display(Markdown(wide_md))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/teshima/lecture-supplement-material/Introduction-to-AI/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/teshima/lecture-supplement-material/Introduction-to-AI/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/teshima/lecture-supplement-material/Introdu

### 品詞タグ付け結果（縦長）

,Token,Lemma,POS,Tag,Dep
0,6,6,NUM,名詞-数詞,compound
1,月,月,NOUN,名詞-普通名詞-助数詞可能,compound
2,27,27,NUM,名詞-数詞,nummod
3,日,日,NOUN,接尾辞-名詞的-助数詞,obl
4,、,、,PUNCT,補助記号-読点,punct
5,ハル,ハル,PROPN,名詞-固有名詞-人名-名,nsubj
6,は,は,ADP,助詞-係助詞,case
7,国立,国立,NOUN,名詞-普通名詞-一般,compound
8,科学,科学,NOUN,名詞-普通名詞-サ変可能,compound
9,博物,博物,NOUN,名詞-普通名詞-一般,compound


### 固有表現認識結果（エンティティ単位）

,Entity,Label,Start,End
0,6月27日,DATE,0,5
1,ハル,PERSON,6,8
2,国立科学博物館,FAC,9,16


### 系列ラベリング結果（横長: 列がトークン、行が属性＋スパン）

|       | 0      | 1      | 2      | 3      | 4     | 5        | 6   | 7     | 8     | 9     | 10    | 11    | 12    | 13    | 14    | 15    | 16    |
|:------|:-------|:-------|:-------|:-------|:------|:---------|:----|:------|:------|:------|:------|:------|:------|:------|:------|:------|:------|
| Token | 6      | 月     | 27     | 日     | 、    | ハル     | は  | 国立  | 科学  | 博物  | 館    | の    | 展示  | を    | 訪れ  | た    | 。    |
| POS   | NUM    | NOUN   | NUM    | NOUN   | PUNCT | PROPN    | ADP | NOUN  | NOUN  | NOUN  | NOUN  | ADP   | NOUN  | ADP   | VERB  | AUX   | PUNCT |
| NER   | B-DATE | I-DATE | I-DATE | I-DATE | O     | B-PERSON | O   | B-FAC | I-FAC | I-FAC | I-FAC | O     | O     | O     | O     | O     | O     |
| Span  | 0-1    | 1-2    | 2-4    | 4-5    | 5-6   | 6-8      | 8-9 | 9-11  | 11-13 | 13-15 | 15-16 | 16-17 | 17-19 | 19-20 | 20-22 | 22-23 | 23-24 |

In [4]:
text = "吾輩は猫である。名前はまだ無い。"
doc = nlp(text)

# -----------------------------------------
# 4. 品詞タグ付け (POS tagging) ― 縦長テーブル
# -----------------------------------------
pos_df = pd.DataFrame({
    "Token": [tok.text for tok in doc],
    "Lemma": [tok.lemma_ for tok in doc],
    "POS":   [tok.pos_   for tok in doc],
    "Tag":   [tok.tag_   for tok in doc],
    "Dep":   [tok.dep_   for tok in doc],
})

display(Markdown("### 品詞タグ付け結果（縦長）"))
display(
    pos_df.style.set_properties(
        **{
            "background-color": "#E0FFFF",  # light_light_blue
            "border-color":     "#D3D3D3",  # light_gray
        }
    )
)

# -----------------------------------------
# 5. 固有表現認識 (NER) ― エンティティ単位
# -----------------------------------------
ner_df = pd.DataFrame({
    "Entity": [ent.text for ent in doc.ents],
    "Label":  [ent.label_ for ent in doc.ents],
    "Start":  [ent.start_char for ent in doc.ents],
    "End":    [ent.end_char   for ent in doc.ents],
})

display(Markdown("### 固有表現認識結果（エンティティ単位）"))
display(ner_df if not ner_df.empty else Markdown("*No named entities detected.*"))

# -----------------------------------------
# 6. 系列ラベリング中間表（Token / POS / NER の縦長）
# -----------------------------------------
seq_df = pd.DataFrame({
    "Token": [tok.text for tok in doc],
    "POS":   [tok.pos_  for tok in doc],
    "NER":   [
        f"{tok.ent_iob_}-{tok.ent_type_}" if tok.ent_type_ else "O"
        for tok in doc
    ],
    # スパンを "start-end" で表現
    "Span": [f"{tok.idx}-{tok.idx+len(tok)}" for tok in doc],
})

# -----------------------------------------
# 7. 横長テーブル（Token が列、行に POS・NER・Span）
#    * スライドに貼り付けやすい形
# -----------------------------------------
wide_df = pd.DataFrame(
    [seq_df["Token"].to_list(),
     seq_df["POS"].to_list(),
     seq_df["NER"].to_list(),
     seq_df["Span"].to_list()],
    index=["Token", "POS", "NER", "Span"]
)

wide_md = wide_df.to_markdown()

display(Markdown("### 系列ラベリング結果（横長: 列がトークン、行が属性＋スパン）"))
display(Markdown(wide_md))

### 品詞タグ付け結果（縦長）

,Token,Lemma,POS,Tag,Dep
0,吾輩,吾輩,PROPN,代名詞,nsubj
1,は,は,ADP,助詞-係助詞,case
2,猫,猫,NOUN,名詞-普通名詞-一般,ROOT
3,で,だ,AUX,助動詞,cop
4,ある,ある,VERB,動詞-非自立可能,fixed
5,。,。,PUNCT,補助記号-句点,punct
6,名前,名前,NOUN,名詞-普通名詞-一般,nsubj
7,は,は,ADP,助詞-係助詞,case
8,まだ,まだ,ADV,副詞,advmod
9,無い,無い,ADJ,形容詞-非自立可能,ROOT


### 固有表現認識結果（エンティティ単位）

*No named entities detected.*

### 系列ラベリング結果（横長: 列がトークン、行が属性＋スパン）

|       | 0     | 1   | 2    | 3   | 4    | 5     | 6    | 7     | 8     | 9     | 10    |
|:------|:------|:----|:-----|:----|:-----|:------|:-----|:------|:------|:------|:------|
| Token | 吾輩  | は  | 猫   | で  | ある | 。    | 名前 | は    | まだ  | 無い  | 。    |
| POS   | PROPN | ADP | NOUN | AUX | VERB | PUNCT | NOUN | ADP   | ADV   | ADJ   | PUNCT |
| NER   | O     | O   | O    | O   | O    | O     | O    | O     | O     | O     | O     |
| Span  | 0-2   | 2-3 | 3-4  | 4-5 | 5-7  | 7-8   | 8-10 | 10-11 | 11-13 | 13-15 | 15-16 |